# Cross-Venue Crypto Arbitrage Simulator

This notebook explains the simulation mechanics and reproduces the main diagnostics.

The goal is to trace the full path from quote events to realized PnL:

```text
market data → signal → latency → execution → inventory → replay → diagnostics

In [ ]:
```python
from pathlib import Path

import pandas as pd

from hft_crypto_arb.plots import make_plot_pack

## 1. Inputs

The simulator writes a trade ledger to:

```text
outputs/trades.csv

In [ ]:

```python
trades_path = Path("../outputs/trades.csv")

if not trades_path.exists():
    raise FileNotFoundError(
        "Run the simulator first, then re-run this notebook. "
        "Expected file: outputs/trades.csv"
    )

trades = pd.read_csv(trades_path)
trades.head()

## 2. Trade ledger

The trade ledger is the main audit object.

Important fields may include:

- signal time
- fill time
- trade direction
- executed quantity
- expected PnL
- realized PnL
- expected edge
- realized edge
- edge decay

The exact columns depend on the current execution model.

In [ ]:
trades.columns.tolist()

## 3. Summary metrics

The first layer of analysis is basic PnL decomposition.

In [ ]:
pnl_col = "realized_pnl" if "realized_pnl" in trades.columns else "pnl"

summary = {
    "n_trades": len(trades),
    "net_pnl": trades[pnl_col].sum(),
    "avg_pnl": trades[pnl_col].mean(),
    "win_rate": (trades[pnl_col] > 0).mean(),
    "gross_profit": trades.loc[trades[pnl_col] > 0, pnl_col].sum(),
    "gross_loss": trades.loc[trades[pnl_col] < 0, pnl_col].sum(),
}

summary

## 4. Diagnostics

The core diagnostic is the difference between signal-time edge and realized edge after latency.

A positive book spread does not guarantee positive realized PnL.

In [ ]:
plot_dir = Path("../outputs/plots")
paths = make_plot_pack(trades, plot_dir)

paths

## 5. Interpretation

The simulator separates signal quality from execution quality.

A quote snapshot may show a positive cross-venue spread. The execution engine then waits for the configured latency interval and fills against the later book.

The difference between the signal-time edge and realized edge measures the cost of latency, adverse selection, and market movement.

Negative results are useful. They show that naive arbitrage can disappear once the execution path is modelled.

## 6. Limitations

This is a research simulator, not a live trading system.

It does not include:

- exchange connectivity
- real order-state management
- queue position
- colocated infrastructure
- persistent audit logs
- reconciliation
- monitoring
- kill switches

## 7. Extensions

Natural next extensions:

- historical L2 data adapter
- venue-specific latency distributions
- legging-risk model
- maker/taker routing
- queue-position model
- inventory rebalancing between venues
- scenario analysis across latency, fees, slippage, and order size